# Building a Dashboard

The explainer introduced us to the thinking that happens before a dashboard is built: choosing actionable metrics over vanity metrics, translating stakeholder goals into measurable indicators, and laying out panels with hierarchy, grouping, and progressive disclosure. A dashboard with 47 metrics is a data dump. A dashboard with five well-chosen metrics is a decision-making tool.

Now we build one. A delegation is arriving in two hours and needs a one-page climate dashboard they can read at a glance. Each panel tells part of the story; together they form a complete briefing. Our job is to make the most important information the most prominent, group related metrics together, and let the layout guide the reader from headline to detail.

By the end of this notebook we will be able to:

- Create **multi-panel figures** using `plt.subplots()`
- Place different chart types in each panel
- Use `plt.suptitle()` for an overall dashboard title
- Apply a **consistent colour scheme** across panels
- Add **annotations** and **trend lines** to highlight key findings

In [ ]:
%pip install -q pandas matplotlib seaborn

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

In [ ]:
temp = pd.read_csv("../data/global_temperature.csv")
energy = pd.read_csv("../data/energy_mix_by_country.csv")
co2 = pd.read_csv("../data/co2_emissions.csv")
renewables = pd.read_csv("../data/renewable_capacity.csv")

# Pre-compute a rolling average for temperature
temp["rolling_10yr"] = temp["anomaly_c"].rolling(window=10, center=True).mean()

---

## 1. Multi-panel figures with `plt.subplots()`

A dashboard is a collection of panels, each answering one question. In matplotlib, `plt.subplots(nrows, ncols)` creates a figure with a grid of axes. Each axis is an independent chart area. This is the technical mechanism behind the grouping principle from the explainer: related charts sit next to each other in the grid.

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 4))

# Left panel
axes[0].plot(temp["year"], temp["anomaly_c"], color="#e6b0aa", linewidth=0.6)
axes[0].plot(temp["year"], temp["rolling_10yr"], color="#c0392b", linewidth=2)
axes[0].set_title("Temperature Anomaly")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Anomaly (\u00b0C)")

# Right panel
co2_2024 = co2[co2["year"] == 2024].sort_values("emissions_mt", ascending=True)
axes[1].barh(co2_2024["country"], co2_2024["emissions_mt"], color="#2980b9")
axes[1].set_title("CO\u2082 Emissions (2024)")
axes[1].set_xlabel("Emissions (Mt)")

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

With one row and two columns, `axes` is a 1D array: `axes[0]` is the left panel, `axes[1]` is the right. When there are multiple rows and columns, `axes` becomes a 2D array: `axes[row, col]`.

### Adding a figure-level title

`plt.suptitle()` adds a title above all panels. In dashboard terms, this is the headline — the first thing the reader sees. The explainer's hierarchy principle says the most important information should be the most visually prominent. The suptitle serves that role at the dashboard level.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(temp["year"], temp["anomaly_c"], color="#e6b0aa", linewidth=0.6)
axes[0].plot(temp["year"], temp["rolling_10yr"], color="#c0392b", linewidth=2)
axes[0].set_title("Temperature Anomaly")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Anomaly (\u00b0C)")

co2_2024 = co2[co2["year"] == 2024].sort_values("emissions_mt", ascending=True)
axes[1].barh(co2_2024["country"], co2_2024["emissions_mt"], color="#2980b9")
axes[1].set_title("CO\u2082 Emissions by Country (2024)")
axes[1].set_xlabel("Emissions (Mt)")

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("Climate Overview", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---

## 2. A 2x2 grid

Four panels give enough space for a proper briefing without overwhelming the reader. Think of this as the explainer's progressive disclosure in action: each panel answers one question, and the reader scans across them to build the full picture.

We will build the dashboard one panel at a time, then combine them. This mirrors a good workflow in practice: get each chart right individually before assembling the whole.

### Panel A: Temperature trend (line chart)

In [ ]:
# Define a colour palette to use consistently
COLOURS = {
    "red": "#c0392b",
    "red_light": "#e6b0aa",
    "blue": "#2471a3",
    "blue_light": "#85c1e9",
    "green": "#27ae60",
    "orange": "#f39c12",
    "yellow": "#f1c40f",
    "teal": "#1abc9c",
    "grey": "#7f8c8d",
}

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(temp["year"], temp["anomaly_c"], color=COLOURS["red_light"], linewidth=0.5)
ax.plot(temp["year"], temp["rolling_10yr"], color=COLOURS["red"], linewidth=2)
ax.axhline(y=0, color="grey", linestyle="--", linewidth=0.5)
ax.set_title("A. Global Temperature Anomaly", fontweight="bold", fontsize=11)
ax.set_xlabel("Year")
ax.set_ylabel("Anomaly (\u00b0C)")

# Annotate the latest value
latest = temp.dropna(subset=["rolling_10yr"]).iloc[-1]
ax.annotate(f"+{latest['rolling_10yr']:.1f}\u00b0C",
            xy=(latest["year"], latest["rolling_10yr"]),
            xytext=(-50, 15), textcoords="offset points",
            fontsize=10, fontweight="bold", color=COLOURS["red"],
            arrowprops=dict(arrowstyle="->", color=COLOURS["red"]))

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

The **annotation** draws attention to the current anomaly, which is the number a policymaker will quote. This is exactly the kind of detail the explainer described when it talked about designing for senior leadership: less data, more interpretation. The title names the metric; the annotation delivers the headline finding.

### Panel B: CO2 emissions trend for major emitters (line chart)

In [ ]:
focus = ["China", "USA", "India", "Germany", "UK"]
focus_colours = [COLOURS["red"], COLOURS["blue"], COLOURS["orange"],
                 COLOURS["green"], COLOURS["teal"]]

fig, ax = plt.subplots(figsize=(6, 4))

for country, c in zip(focus, focus_colours):
    subset = co2[co2["country"] == country]
    ax.plot(subset["year"], subset["emissions_mt"], color=c, linewidth=1.5, label=country)

ax.set_title("B. CO\u2082 Emissions Trend", fontweight="bold", fontsize=11)
ax.set_xlabel("Year")
ax.set_ylabel("Emissions (Mt CO\u2082)")
ax.legend(fontsize=8, loc="upper left")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

### Panel C: Renewable capacity growth (stacked area)

The delegation will want to know whether renewable energy is growing fast enough. A stacked area chart shows both the total and its components over time, which makes it a natural fit for a **composition + trend** question.

In [ ]:
# Aggregate global renewable capacity by year
global_ren = renewables.groupby("year")[["solar_gw", "wind_gw", "hydro_gw"]].sum()

fig, ax = plt.subplots(figsize=(6, 4))

ax.stackplot(global_ren.index,
             global_ren["solar_gw"], global_ren["wind_gw"], global_ren["hydro_gw"],
             labels=["Solar", "Wind", "Hydro"],
             colors=[COLOURS["yellow"], COLOURS["blue_light"], COLOURS["teal"]],
             alpha=0.85)

ax.set_title("C. Global Renewable Capacity", fontweight="bold", fontsize=11)
ax.set_xlabel("Year")
ax.set_ylabel("Installed capacity (GW)")
ax.legend(loc="upper left", fontsize=8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

A **stacked area chart** shows how the total and its components grow together. Hydro is relatively flat at the bottom; solar is the fast-growing layer on top. The reader gets both the composition story and the trend story in a single panel.

### Panel D: Per-capita emissions by country (bar chart)

The final panel shifts from trend to comparison. Per-capita emissions answer a different question from total emissions, and the distinction matters for policy. A horizontal bar chart sorted by value is the clearest way to present this ranking.

In [ ]:
pc_2024 = co2[co2["year"] == 2024].sort_values("per_capita_t", ascending=True)

fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.barh(pc_2024["country"], pc_2024["per_capita_t"], color=COLOURS["blue"])

# Highlight the highest bar
bars[-1].set_color(COLOURS["red"])

ax.set_title("D. Per-Capita CO\u2082 Emissions (2024)", fontweight="bold", fontsize=11)
ax.set_xlabel("Tonnes CO\u2082 per person")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

Highlighting the top bar in red draws the eye to the country with the highest per-capita emissions. This is a simple use of colour as a secondary encoding to direct attention rather than encode data.

---

## 3. Combining all four panels

Now we assemble the full dashboard. The design decisions here map directly to the layout principles from the explainer:

- **Hierarchy**: the `suptitle` delivers the headline. Panel A (temperature) sits top-left, where the reader's eye lands first.
- **Grouping**: the top row shows trends over time; the bottom row shows capacity and per-capita snapshots.
- **Consistency**: the `COLOURS` dictionary ensures the same red, blue, and teal appear across all panels, tying the dashboard together visually.
- **`figsize=(14, 10)`** gives enough room for four panels on one page.
- **`plt.tight_layout()`** with `rect` parameter leaves space for the suptitle.

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 10))

# ----- Panel A: Temperature anomaly (top left) -----
ax = axes[0, 0]
ax.plot(temp["year"], temp["anomaly_c"], color=COLOURS["red_light"], linewidth=0.5)
ax.plot(temp["year"], temp["rolling_10yr"], color=COLOURS["red"], linewidth=2)
ax.axhline(y=0, color="grey", linestyle="--", linewidth=0.5)
ax.set_title("A. Global Temperature Anomaly", fontweight="bold", fontsize=11)
ax.set_xlabel("Year")
ax.set_ylabel("Anomaly (\u00b0C)")
latest = temp.dropna(subset=["rolling_10yr"]).iloc[-1]
ax.annotate(f"+{latest['rolling_10yr']:.1f}\u00b0C",
            xy=(latest["year"], latest["rolling_10yr"]),
            xytext=(-50, 15), textcoords="offset points",
            fontsize=10, fontweight="bold", color=COLOURS["red"],
            arrowprops=dict(arrowstyle="->", color=COLOURS["red"]))

# ----- Panel B: CO2 emissions trend (top right) -----
ax = axes[0, 1]
for country, c in zip(focus, focus_colours):
    subset = co2[co2["country"] == country]
    ax.plot(subset["year"], subset["emissions_mt"], color=c, linewidth=1.5, label=country)
ax.set_title("B. CO\u2082 Emissions by Major Emitter", fontweight="bold", fontsize=11)
ax.set_xlabel("Year")
ax.set_ylabel("Emissions (Mt CO\u2082)")
ax.legend(fontsize=8, loc="upper left")

# ----- Panel C: Renewable capacity (bottom left) -----
ax = axes[1, 0]
ax.stackplot(global_ren.index,
             global_ren["solar_gw"], global_ren["wind_gw"], global_ren["hydro_gw"],
             labels=["Solar", "Wind", "Hydro"],
             colors=[COLOURS["yellow"], COLOURS["blue_light"], COLOURS["teal"]],
             alpha=0.85)
ax.set_title("C. Global Renewable Capacity", fontweight="bold", fontsize=11)
ax.set_xlabel("Year")
ax.set_ylabel("Installed capacity (GW)")
ax.legend(loc="upper left", fontsize=8)

# ----- Panel D: Per-capita emissions (bottom right) -----
ax = axes[1, 1]
pc_2024 = co2[co2["year"] == 2024].sort_values("per_capita_t", ascending=True)
bars = ax.barh(pc_2024["country"], pc_2024["per_capita_t"], color=COLOURS["blue"])
bars[-1].set_color(COLOURS["red"])
ax.set_title("D. Per-Capita CO\u2082 (2024)", fontweight="bold", fontsize=11)
ax.set_xlabel("Tonnes CO\u2082 per person")

# ----- Formatting -----
for ax in axes.flat:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("Climate and Energy Dashboard",
             fontsize=18, fontweight="bold", y=1.01)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

### What ties the dashboard together

This is where the explainer's principles become tangible:

- **Shared colour palette**: the same red appears in the temperature panel and the highlighted bar. The same blue family appears in the emissions and capacity panels. Consistency reduces the reader's cognitive load.
- **Panel labels** (A, B, C, D) let the briefer reference specific panels during discussion: "If you look at Panel C, you can see the acceleration in solar since 2015."
- **Consistent axis formatting**: all panels have clean spines and clear labels. No chartjunk.
- **One story per panel**: each chart answers one question. Together, they tell a coherent narrative — temperatures are rising, emissions are shifting between countries, renewables are growing, and per-capita burdens vary widely. That is a dashboard built around stakeholder questions, not around available data.

---

## 4. Adding trend lines

A **trend line** (linear regression line) summarises the overall direction of noisy data. This is another form of the annotation principle from the explainer: it adds interpretation to the chart so the reader does not have to estimate the trend by eye. NumPy's `polyfit` fits a straight line.

In [ ]:
# Fit a linear trend to the temperature data from 1970 onwards
recent = temp[temp["year"] >= 1970].copy()
coeffs = np.polyfit(recent["year"], recent["anomaly_c"], 1)
trend_line = np.poly1d(coeffs)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(temp["year"], temp["anomaly_c"], color=COLOURS["red_light"], linewidth=0.6)
ax.plot(recent["year"], trend_line(recent["year"]),
        color=COLOURS["red"], linewidth=2, linestyle="--",
        label=f"Trend: +{coeffs[0]:.3f}\u00b0C/year")
ax.axhline(y=0, color="grey", linestyle="--", linewidth=0.5)
ax.set_title("Temperature Anomaly with Linear Trend (1970\u2013present)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Anomaly (\u00b0C)")
ax.legend(fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

The legend includes the slope, so the reader immediately knows the rate of warming. For a senior audience, that single number is more useful than the entire scatter of annual points. This is designing for the audience: giving the delegation the figure they will cite in their own meetings.

---

## 5. Unequal panel sizes with `gridspec_kw`

Sometimes one question deserves more visual space than another. The `gridspec_kw` parameter controls the relative sizes of rows and columns, letting us apply the hierarchy principle at the panel level.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4),
                         gridspec_kw={"width_ratios": [2, 1]})

# Wide panel: line chart
axes[0].plot(temp["year"], temp["anomaly_c"], color=COLOURS["red_light"], linewidth=0.5)
axes[0].plot(temp["year"], temp["rolling_10yr"], color=COLOURS["red"], linewidth=2)
axes[0].set_title("Temperature Anomaly (full history)")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Anomaly (\u00b0C)")

# Narrow panel: bar chart
pc_2024 = co2[co2["year"] == 2024].sort_values("per_capita_t", ascending=True)
axes[1].barh(pc_2024["country"], pc_2024["per_capita_t"], color=COLOURS["blue"])
axes[1].set_title("Per-Capita CO\u2082 (2024)")
axes[1].set_xlabel("Tonnes per person")

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

The line chart gets twice the width of the bar chart because it spans 175 years of data, while the bar chart only shows a snapshot. Giving the more complex panel more space is a practical application of the hierarchy principle: the most important or information-dense metric gets the most room.

---

## Exercises

These exercises ask you to build your own dashboards. For each one, think about the questions the reader needs answered and apply the layout principles: hierarchy, grouping, and consistency.

### Exercise 1: Two-panel energy comparison

Create a figure with 1 row and 2 columns. In the left panel, show a stacked bar chart of the UK's energy mix in 2024 (one bar per source). In the right panel, show the same for China. Use the same colour for each energy source in both panels so they can be compared directly. Add a `suptitle`.

In [ ]:
# Your code here


### Exercise 2: Three-panel renewable growth

Create a figure with 1 row and 3 columns: one panel each for solar, wind, and hydro capacity. In each panel, plot lines for China, USA, India, and Germany (2010-2024). Use the same colour for each country across all three panels. Add a `suptitle` and ensure all y-axes start at zero.

In [ ]:
# Your code here


### Exercise 3: Full four-panel dashboard

Build your own 2x2 dashboard. Choose four charts that together tell a story about one country's climate and energy situation. Before you write any code, decide: what are the four questions this dashboard should answer? Who is the reader? What should they see first?

Requirements:

- Use at least two different chart types (line, bar, scatter, histogram, or stacked area)
- Use a consistent colour palette defined as a dictionary
- Add a `suptitle` and panel labels (A, B, C, D)
- Include at least one annotation or trend line
- Remove top and right spines from all panels

In [ ]:
# Your code here


---

## Summary

We have built the kind of dashboard the explainer described: one where every panel answers a specific question and the layout guides the reader from headline to detail. Specifically, we:

- Created **multi-panel figures** with `plt.subplots(nrows, ncols)`
- Used `plt.suptitle()` to give the dashboard a headline
- Built up a dashboard **panel by panel**, then assembled the complete figure
- Applied a **consistent colour palette** across all panels using a shared dictionary
- Added **annotations** with arrows to highlight the findings a policymaker would cite
- Fitted **trend lines** with `np.polyfit()` to summarise noisy data into actionable numbers
- Controlled **panel sizes** with `gridspec_kw` to give more space to more important metrics

A good dashboard is not a collection of charts. It is an argument, structured visually. Each panel answers one question, and together they give the reader a complete picture without requiring them to ask "what am I looking at?"